# Perplexity and selective LLM review

Minerva scores each passage for perplexity. Qwen reviews selected passages, and the pipeline validates the returned spans against the original text before merging them with the first-pass candidates.

Run the setup cells, then enable the corpus run or the optional reference evaluation. Corpus extraction only requires the first-pass JSON. Reference evaluation also requires a matching set of human-reviewed annotations.


## Set up the environment

For new inference, select a **T4 GPU runtime** and enable the `HF_TOKEN` Colab Secret. The token needs read access to `sapienzanlp/Minerva-350M-base-v1.0` on Hugging Face. A CPU runtime is sufficient when evaluating saved outputs.

Upload `baseline.json` from the [rules and spaCy notebook](rules_and_spacy_ner_colab.ipynb) to `/content/`, or change the input path below to point to a saved first-pass result. To reproduce the recorded thesis experiments, use the original first-pass snapshot described in the README.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

repo = Path("/content/forensic-llm-graph")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/AntoFratta/forensic-llm-graph.git", str(repo)],
        check=True,
    )

os.chdir(repo)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
input_path = Path("/content/baseline.json")
if not input_path.is_file():
    raise FileNotFoundError(f"Upload {input_path.name} to {input_path.parent} before continuing.")


In [ ]:
def model_environment():
    """Check Colab GPU and model access before launching inference."""
    import torch
    from google.colab import userdata
    from huggingface_hub import hf_hub_download

    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime in Colab before starting inference.")

    try:
        token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        raise RuntimeError("Add the HF_TOKEN Secret and enable access for this notebook.") from None
    if not token:
        raise RuntimeError("Set a Hugging Face read token in the HF_TOKEN Secret.")

    with Path("configs/hybrid.json").open(encoding="utf-8") as settings:
        minerva = json.load(settings)["perplexity"]
    hf_hub_download(minerva["model"], "config.json", revision=minerva["revision"], token=token)
    print(f"GPU: {torch.cuda.get_device_name(0)}. Minerva access verified.")
    return {**os.environ, "HF_TOKEN": token}


def run_hybrid(output_path, options=(), resume=False):
    """Run the hybrid script and stream progress to Colab."""
    command = [
        sys.executable, "-u", "scripts/run_hybrid.py",
        "--input", str(input_path), "--output", str(output_path),
        *options,
    ]
    if resume:
        command.append("--resume")

    with subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env=model_environment(),
    ) as process:
        try:
            for line in process.stdout:
                if line.startswith("Perplexity:"):
                    done, total = map(int, line.split(":", 1)[1].split("/"))
                    if total > 100 and done % 100 and done != total:
                        continue
                print(line, end="", flush=True)
            return_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise

    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


## Run extraction on the full corpus

Set `run_corpus` to `True` to start. Minerva scores every passage and Qwen reviews up to 100 under the configured budget. Allow about an hour on a T4 for the full corpus with cached weights, plus time for the initial model downloads.

Download `outputs/hybrid_corpus.json` before disconnecting. To resume an interrupted run, keep the companion `.checkpoints.jsonl` file and use the same input, code, settings and output path with `resume=True`.


In [ ]:
run_corpus = False
corpus_output = Path("outputs/hybrid_corpus.json")

if run_corpus:
    run_hybrid(corpus_output, resume=False)
    print(f"Download {corpus_output} from the Colab Files panel.")


## Evaluate routing on reviewed passages (optional)

Use a matching human-reviewed reference JSON and set `run_reference_evaluation` to `True`. The thesis annotations are kept outside Git. Reference labels are used only for scoring.

To evaluate saved predictions on CPU, point `reference_output` to the existing model JSON and leave `run_corpus` disabled. For a new comparison, enable `generate_reference_predictions` in a T4 runtime. Each routing policy selects passages under the same budget, and Qwen reviews the union once. The report gives overall exact-span precision, recall and F1.


In [ ]:
run_reference_evaluation = False
generate_reference_predictions = False
reference_path = Path("/content/test_reference.reviewed.json")
reference_output = Path("outputs/hybrid_reference.json")
evaluation_output = Path("outputs/reference_evaluation.json")

if run_reference_evaluation:
    if not reference_path.is_file():
        raise FileNotFoundError(f"Upload {reference_path.name} or set the reference path.")
    if generate_reference_predictions:
        run_hybrid(
            reference_output,
            options=("--reference", str(reference_path), "--compare-routing"),
            resume=False,
        )
    elif not reference_output.is_file():
        raise FileNotFoundError("Set reference_output to a saved predictions file or enable generation.")

    subprocess.run(
        [
            sys.executable, "scripts/evaluate.py",
            "--reference", str(reference_path), "--baseline", str(input_path),
            "--hybrid", str(reference_output), "--output", str(evaluation_output),
        ],
        check=True,
    )
    print(f"Download {reference_output} and {evaluation_output} from the Colab Files panel.")


## Score an existing corpus output (optional)

Set `score_corpus` to `True` to measure exact-span scores on the annotated passages using the full-corpus predictions. Set `corpus_output` and `reference_path` to the saved files and leave `run_corpus` disabled to evaluate on CPU. These scores describe what happened to the reference passages under the global review budget.


In [ ]:
score_corpus = False
corpus_evaluation_output = Path("outputs/corpus_reference_evaluation.json")

if score_corpus:
    if not corpus_output.is_file() or not reference_path.is_file():
        raise FileNotFoundError("Provide the full-corpus output and a matching reviewed reference.")
    if corpus_evaluation_output.exists():
        raise FileExistsError("Choose a new path for the corpus evaluation output.")

    import hashlib

    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from scripts.evaluate import overall_metrics
    from scripts.run_hybrid import write_result
    from src.evaluation import load_reference, score_reference

    reference, reference_hash = load_reference(reference_path)
    with corpus_output.open(encoding="utf-8") as result:
        corpus = json.load(result)
    run = corpus["run"]
    if (run["stage"] != "hybrid" or run["scope"] != "extended"
            or len(corpus["documents"]) != run["input_document_count"]):
        raise ValueError("Expected a complete full-corpus hybrid output.")
    if run["input_sha256"] != hashlib.sha256(input_path.read_bytes()).hexdigest():
        raise ValueError("The corpus output was generated from a different first-pass input.")

    metrics = score_reference(reference, corpus["documents"])
    metrics["overall"] = overall_metrics(metrics)
    write_result(corpus_evaluation_output, {
        "stage": "full_corpus_reference_evaluation",
        "reference_sha256": reference_hash,
        "model_output_sha256": hashlib.sha256(corpus_output.read_bytes()).hexdigest(),
        "run": run,
        "routing": corpus["routing"],
        "metrics": metrics,
    })
    print(f"Overall metrics: {metrics['overall']}")
    print(f"Download {corpus_evaluation_output} from the Colab Files panel.")
